# YOLOv8 for Tokamak Plasma Detection

This notebook trains and evaluates YOLOv8 for detecting plasma in tokamak images.

## 1. Install Dependencies

In [ ]:
!pip install ultralytics

## 2. Import Libraries

In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch
import numpy as np
from ultralytics import YOLO

# Add ingestion_program to path
sys.path.insert(0, 'ingestion_program')
from tokam2d_utils import TokamDataset

# Import YOLO training functions
sys.path.insert(0, 'solution')
from train_model_yolo import prepare_yolo_dataset, train_yolo_model, evaluate_yolo_model, predict_with_yolo

## 3. Load Dataset

Load the tokamak dataset with bounding box annotations.

In [ ]:
# Specify your data directory
data_dir = '../tokam_data'  # Adjust this path as needed

# Load dataset
dataset = TokamDataset(data_dir, include_unlabeled=True)
print(f"Total samples: {len(dataset)}")

# Count labeled vs unlabeled
labeled_count = sum(1 for i in range(len(dataset)) if dataset[i][1]['boxes'] is not None and len(dataset[i][1]['boxes']) > 0)
print(f"Labeled samples: {labeled_count}")
print(f"Unlabeled samples: {len(dataset) - labeled_count}")

## 4. Visualize Sample Data

Display a few samples with their bounding boxes.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx in range(6):
    image, target = dataset[idx]
    img_np = image.squeeze().numpy()
    
    axes[idx].imshow(img_np, cmap='hot')
    axes[idx].axis('off')
    
    # Draw bounding boxes if present
    if target['boxes'] is not None and len(target['boxes']) > 0:
        for box in target['boxes']:
            x, y, w, h = box.numpy()
            x_pixel = x * 512
            y_pixel = y * 512
            w_pixel = w * 512
            h_pixel = h * 512
            rect = patches.Rectangle((x_pixel, y_pixel), w_pixel, h_pixel,
                                    linewidth=2, edgecolor='lime', facecolor='none')
            axes[idx].add_patch(rect)
        axes[idx].set_title(f'Sample {idx} (labeled)')
    else:
        axes[idx].set_title(f'Sample {idx} (unlabeled)')

plt.tight_layout()
plt.show()

## 5. Prepare YOLO Dataset

Convert the dataset to YOLO format (images + label txt files).

In [ ]:
# Prepare dataset in YOLO format
dataset_yaml_path = prepare_yolo_dataset(
    data_dir=data_dir,
    output_dir='yolo_tokam',
    val_split=0.2
)

print(f"\nDataset prepared at: {dataset_yaml_path}")
print("\nDataset structure:")
print("yolo_tokam/")
print("  ├── images/")
print("  │   ├── train/")
print("  │   └── val/")
print("  ├── labels/")
print("  │   ├── train/")
print("  │   └── val/")
print("  └── dataset.yaml")

## 6. Train YOLOv8 Model

Train the model with augmentation. This may take some time depending on your hardware.

In [ ]:
# Train the model
model = train_yolo_model(
    dataset_yaml=dataset_yaml_path,
    model_size='n',  # Options: 'n', 's', 'm', 'l', 'x' (nano to extra-large)
    epochs=50,
    batch_size=16,
    img_size=512,
    device=0 if torch.cuda.is_available() else 'cpu'
)

print("\nTraining complete!")
print(f"Best weights saved at: runs/detect/train/weights/best.pt")

## 7. Evaluate on Validation Set

Compute mAP, precision, and recall metrics.

In [ ]:
# Evaluate the model
results = evaluate_yolo_model(
    model_path='runs/detect/train/weights/best.pt',
    dataset_yaml=dataset_yaml_path
)

print("\nValidation Metrics:")
print(f"mAP@0.5: {results['mAP50']:.4f}")
print(f"mAP@0.5:0.95: {results['mAP50-95']:.4f}")
print(f"Precision: {results['precision']:.4f}")
print(f"Recall: {results['recall']:.4f}")

## 8. Visualize Training Results

YOLOv8 automatically saves training plots. Let's display them.

In [ ]:
from IPython.display import Image, display

# Display training results
results_path = Path('runs/detect/train')

plots = ['results.png', 'confusion_matrix.png', 'F1_curve.png', 'PR_curve.png']

for plot_name in plots:
    plot_path = results_path / plot_name
    if plot_path.exists():
        print(f"\n{plot_name}:")
        display(Image(filename=str(plot_path)))
    else:
        print(f"\n{plot_name} not found")

## 9. Visualize Predictions on Validation Set

Compare ground truth boxes (green) with predictions (red).

In [ ]:
# Load the trained model
model = YOLO('runs/detect/train/weights/best.pt')

# Load validation images
val_images_dir = Path('yolo_tokam/images/val')
val_labels_dir = Path('yolo_tokam/labels/val')

# Get list of validation images
val_image_files = sorted(list(val_images_dir.glob('*.png')))[:6]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, img_path in enumerate(val_image_files):
    # Load image
    img = plt.imread(img_path)
    
    # Get predictions
    results = model(img_path, verbose=False)
    
    # Plot image
    axes[idx].imshow(img, cmap='hot')
    axes[idx].axis('off')
    
    # Load ground truth labels
    label_path = val_labels_dir / f"{img_path.stem}.txt"
    if label_path.exists():
        with open(label_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 5:
                    _, x_center, y_center, w, h = map(float, parts)
                    # Convert from normalized YOLO format to pixel coordinates
                    x_pixel = (x_center - w/2) * 512
                    y_pixel = (y_center - h/2) * 512
                    w_pixel = w * 512
                    h_pixel = h * 512
                    rect = patches.Rectangle((x_pixel, y_pixel), w_pixel, h_pixel,
                                            linewidth=2, edgecolor='green', facecolor='none')
                    axes[idx].add_patch(rect)
    
    # Draw predicted boxes
    if len(results[0].boxes) > 0:
        for box in results[0].boxes:
            xyxy = box.xyxy[0].cpu().numpy()
            x1, y1, x2, y2 = xyxy
            w = x2 - x1
            h = y2 - y1
            conf = box.conf[0].cpu().numpy()
            
            rect = patches.Rectangle((x1, y1), w, h,
                                    linewidth=2, edgecolor='red', facecolor='none',
                                    linestyle='--')
            axes[idx].add_patch(rect)
            axes[idx].text(x1, y1-5, f'{conf:.2f}', color='red', fontsize=10, weight='bold')
    
    axes[idx].set_title(f'Val Sample {idx}')

# Add legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='none', edgecolor='green', linewidth=2, label='Ground Truth'),
    Patch(facecolor='none', edgecolor='red', linewidth=2, linestyle='--', label='Prediction')
]
fig.legend(handles=legend_elements, loc='upper center', ncol=2, bbox_to_anchor=(0.5, 0.98))

plt.tight_layout()
plt.show()

## 10. Test Data Inference

Generate predictions on test turb_i.h5 files.

In [ ]:
# Predict on test files
test_files = ['../tokam_data/turb_0.h5', '../tokam_data/turb_1.h5']  # Adjust paths as needed

test_results = predict_with_yolo(
    model_path='runs/detect/train/weights/best.pt',
    h5_files=test_files,
    output_dir='yolo_test_predictions'
)

print(f"\nProcessed {len(test_results)} test files")
print(f"Results saved to: yolo_test_predictions/")

# Display summary
for file_path, predictions in test_results.items():
    print(f"\n{file_path}: {len(predictions)} predictions")

## 11. Visualize Test Predictions

Display predictions on a few test samples.

In [ ]:
import h5py

# Load one test file and visualize predictions
test_file = test_files[0]
predictions = test_results[test_file]

# Load test images
with h5py.File(test_file, 'r') as f:
    # Try different possible keys
    key = None
    for possible_key in ['rho', 'images', 'data']:
        if possible_key in f:
            key = possible_key
            break
    if key is None:
        key = list(f.keys())[0]
    
    images = f[key][:]

# Visualize 6 samples
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx in range(min(6, len(images))):
    img = images[idx]
    axes[idx].imshow(img, cmap='hot')
    axes[idx].axis('off')
    
    # Draw predicted boxes
    if idx < len(predictions):
        pred = predictions[idx]
        if pred['boxes'] is not None and len(pred['boxes']) > 0:
            for box_idx, box in enumerate(pred['boxes']):
                x, y, w, h = box
                conf = pred['confidences'][box_idx]
                
                rect = patches.Rectangle((x, y), w, h,
                                        linewidth=2, edgecolor='red', facecolor='none')
                axes[idx].add_patch(rect)
                axes[idx].text(x, y-5, f'{conf:.2f}', color='red', fontsize=10, weight='bold',
                             bbox=dict(boxstyle='round,pad=0.3', facecolor='black', alpha=0.5))
    
    axes[idx].set_title(f'Test Sample {idx}')

plt.tight_layout()
plt.show()